# Taller abandono de producto financiero

In [1]:
%pip install pandas numpy matplotlib statsmodels scipy

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -------------------------- ------------- 6.3/9.5 MB 38.0 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 32.3 MB/s  0:00:00
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------  11.3/11.3 MB 98.6 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 49.5 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   --------------------- ------------------ 19.9/37.4 MB 95.0 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 92.9 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 92.9 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 48.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 38.6 MB/s  0:00:00

   --- ---------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency

plt.rcParams["figure.figsize"] = (6, 4)

CSV = "abandono_producto_financiero.csv"

In [3]:
datos = pd.read_csv(CSV)
datos.head()

,numero_fila,id_cliente,apellido,puntaje_crediticio,pais,sexo,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
0,1,15634602,Hargrave,619,Francia,Mujer,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,España,Mujer,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,Francia,Mujer,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,Francia,Mujer,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,España,Mujer,43,2,125510.82,1,1,1,79084.10,0


In [4]:
datos.columns.tolist()

['numero_fila',
 'id_cliente',
 'apellido',
 'puntaje_crediticio',
 'pais',
 'sexo',
 'edad',
 'antiguedad',
 'saldo',
 'numero_productos',
 'tiene_tarjeta',
 'miembro_activo',
 'salario_estimado',
 'abandono']

In [5]:
datos.shape

(10000, 14)

In [6]:
datos.dtypes

numero_fila             int64
id_cliente              int64
apellido                  str
puntaje_crediticio      int64
pais                      str
sexo                      str
edad                    int64
antiguedad              int64
saldo                 float64
numero_productos        int64
tiene_tarjeta           int64
miembro_activo          int64
salario_estimado      float64
abandono                int64
dtype: object

In [7]:
datos.describe()

,numero_fila,id_cliente,puntaje_crediticio,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [8]:
# Quitar identificadores: no explican el abandono, solo identifican al cliente
datos = datos.drop(columns=["numero_fila", "id_cliente", "apellido"])

# Factores con los niveles del diccionario; el primero queda como referencia
datos["pais"] = pd.Categorical(datos["pais"], categories=["Francia", "Alemania", "España"])
datos["sexo"] = pd.Categorical(datos["sexo"], categories=["Mujer", "Hombre"])

# La respuesta como entero 0/1
datos["abandono"] = datos["abandono"].astype(int)

print(datos.shape)
datos.dtypes

(10000, 11)


puntaje_crediticio       int64
pais                  category
sexo                  category
edad                     int64
antiguedad               int64
saldo                  float64
numero_productos         int64
tiene_tarjeta            int64
miembro_activo           int64
salario_estimado       float64
abandono                 int64
dtype: object

### R. Punto 1

1. La base original tiene 14 variables. Quitamos 3 (numero_fila, id_cliente y apellido) porque solo sirven para identificar al cliente y no explican el abandono. Para el modelo quedan 11: 10 explicativas y la variable respuesta (abandono).

2. Son 6 cuantitativas: puntaje_crediticio, edad, antiguedad, saldo, numero_productos y salario_estimado.

3. Son 5: dos categóricas (pais y sexo) y tres binarias de 0 y 1 (tiene_tarjeta, miembro_activo y abandono). Sumadas con las 6 cuantitativas dan las 11 variables.

4. El modelo solo entiende números. Las binarias entran directo porque ya vienen en 0 y 1, pero país y sexo son texto y hay que convertirlas. Con C(pais) y C(sexo) Python lo hace solo: deja a Francia y a Mujer como base de comparación y crea las dummies de las demás categorías. Hacerlas a mano da más trabajo y es fácil equivocarse, por ejemplo metiendo una dummy de más, lo que daña el modelo porque la información queda repetida.

# Punto 2